Missing Data – Final Project
CS 509 – Pattern Recognition
Author: Ahmad (Solo Project)

This notebook implements and compares six methods to handle missing data:

Imputation by a Single Value

Imputation by Center of the Group

Imputation from k Nearest Neighbors (k-NN)

Imputation by Partial Mean

Imputation by Singular Value Decomposition (SVD)

EM Algorithm-Based Imputation

For each method, we provide:

The principle (explained in the PPT)

A simple example dataset

A Python implementation

Comparison across all methods

🧩 Purpose of the Notebook

This notebook contains all the programmed solutions required for the final project.
All results, implementations, and visual interpretations are included here.

🧪 Dataset

We will create a small synthetic dataset with missing values to demonstrate and compare different imputation approaches.

In [1]:
# Missing Data – Final Project (CS 509 – Pattern Recognition)
# Author: Ahmad

import numpy as np
import pandas as pd

# For k-NN imputation
from sklearn.impute import KNNImputer

# For SVD imputation
from sklearn.decomposition import TruncatedSVD

# For EM imputation
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

# For handling warnings
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully.")


All libraries loaded successfully.


2. Create a Synthetic Dataset with Missing Values

In this section, we create a simple dataset to demonstrate and compare all imputation methods.
The dataset includes:

Numeric features

A group/category column

Randomly introduced missing values (NaN)

This allows us to test each imputation technique under controlled conditions.

In [2]:
# Create a synthetic dataset with missing values

np.random.seed(0)

data = pd.DataFrame({
    "feature_1": [1.0, 2.0, np.nan, 4.0, 5.0, np.nan],
    "feature_2": [2.5, np.nan, 3.5, 4.5, 5.5, 6.5],
    "feature_3": [10, 11, 12, np.nan, 14, 15],
    "group":     ["A", "A", "B", "B", "B", "A"]
})

print("Original dataset with missing values:")
data


Original dataset with missing values:


,feature_1,feature_2,feature_3,group
0,1.0,2.5,10.0,A
1,2.0,NaN,11.0,A
2,NaN,3.5,12.0,B
3,4.0,4.5,NaN,B
4,5.0,5.5,14.0,B
5,NaN,6.5,15.0,A


3. Imputation by a Single Value (Mean, Median, Default)

This method replaces missing values using a fixed constant, such as:

Mean of the feature

Median of the feature

Mode (most frequent value)

A predefined constant (e.g., 0)

It is the simplest and fastest imputation method, but it does not capture relationships between features.

In [3]:
# Method 1: Imputation by a Single Value (Mean / Median)

data_single = data.copy()

numeric_cols = ["feature_1", "feature_2", "feature_3"]

# Mean imputation
data_mean = data_single.copy()
for col in numeric_cols:
    data_mean[col] = data_mean[col].fillna(data_mean[col].mean())

print("Mean Imputation:")
display(data_mean)

# Median imputation
data_median = data_single.copy()
for col in numeric_cols:
    data_median[col] = data_median[col].fillna(data_median[col].median())

print("\nMedian Imputation:")
display(data_median)


Mean Imputation:


,feature_1,feature_2,feature_3,group
0,1.0,2.5,10.0,A
1,2.0,4.5,11.0,A
2,3.0,3.5,12.0,B
3,4.0,4.5,12.4,B
4,5.0,5.5,14.0,B
5,3.0,6.5,15.0,A



Median Imputation:


,feature_1,feature_2,feature_3,group
0,1.0,2.5,10.0,A
1,2.0,4.5,11.0,A
2,3.0,3.5,12.0,B
3,4.0,4.5,12.0,B
4,5.0,5.5,14.0,B
5,3.0,6.5,15.0,A


4. Imputation by Center of the Group (Group Mean)

This method imputes missing values using the mean of the group that each sample belongs to.

Examples of groups:

Classes (A, B, C)

Categories

Clusters

It captures local structure because each group receives its own imputation value.

In [4]:
# Method 2: Imputation by Center of the Group

data_group = data.copy()

# Group-wise mean imputation
for col in numeric_cols:
    data_group[col] = data_group.groupby("group")[col].transform(
        lambda group_values: group_values.fillna(group_values.mean())
    )

print("Group-Centered Imputation:")
display(data_group)


Group-Centered Imputation:


,feature_1,feature_2,feature_3,group
0,1.0,2.5,10.0,A
1,2.0,4.5,11.0,A
2,4.5,3.5,12.0,B
3,4.0,4.5,13.0,B
4,5.0,5.5,14.0,B
5,1.5,6.5,15.0,A


5. Imputation using k Nearest Neighbors (k-NN)

This method imputes missing values based on the k most similar samples in the dataset.

Steps:

Compute distances between samples

Identify the k nearest neighbors

Replace missing values using the neighbors’ values

Mean of neighbors

Or weighted mean

k-NN considers relationships between features and often gives better results than simple mean/median imputation.

In [5]:
# Method 3: Imputation using k-Nearest Neighbors (k-NN)

data_knn = data.copy()

# Drop non-numeric column for KNNImputer
X = data_knn[numeric_cols]

# Initialize k-NN imputer
knn_imputer = KNNImputer(n_neighbors=3)

# Fit & transform
X_imputed = knn_imputer.fit_transform(X)

# Put results back into dataframe
data_knn[numeric_cols] = X_imputed

print("k-NN Imputation (k=3):")
display(data_knn)


k-NN Imputation (k=3):


,feature_1,feature_2,feature_3,group
0,1.000000,2.5,10.000000,A
1,2.000000,3.5,11.000000,A
2,2.333333,3.5,12.000000,B
3,4.000000,4.5,12.333333,B
4,5.000000,5.5,14.000000,B
5,3.666667,6.5,15.000000,A


6. Imputation by Partial Mean

Partial mean imputation replaces missing values using the mean computed from a selected subset of the data, rather than the entire feature.

Examples of partial subsets:

Removing outliers

Using only values within a defined range

Using only non-zero values

Using domain-specific filters

This method avoids distortion from extreme values and can better represent the true central tendency.

In [6]:
# Method 4: Imputation by Partial Mean (example: removing outliers)

from scipy.stats import zscore

data_partial = data.copy()

# Compute z-scores for numeric columns
z = zscore(data_partial[numeric_cols], nan_policy='omit')

# Keep rows where all numeric features are non-outliers (|z| < 2)
partial_subset = data_partial[(np.abs(z) < 2).all(axis=1)][numeric_cols]

# Compute partial means
partial_means = partial_subset.mean()

# Impute missing values with partial means
data_partial_imputed = data.copy()
for col in numeric_cols:
    data_partial_imputed[col] = data_partial_imputed[col].fillna(partial_means[col])

print("Partial Mean Imputation (outliers removed before mean):")
display(data_partial_imputed)


Partial Mean Imputation (outliers removed before mean):


,feature_1,feature_2,feature_3,group
0,1.0,2.5,10.0,A
1,2.0,4.0,11.0,A
2,3.0,3.5,12.0,B
3,4.0,4.5,12.0,B
4,5.0,5.5,14.0,B
5,3.0,6.5,15.0,A


7. Imputation by Singular Value Decomposition (SVD)

SVD decomposes the dataset into latent components:

X≈UΣV
T

Missing values can be estimated by:

Initializing missing values (e.g., mean imputation)

Applying SVD to learn low-rank structure

Reconstructing the matrix

Updating missing values using the reconstruction

SVD works especially well when the dataset has a strong linear or low-rank structure.

In [7]:
# Method 5: Imputation using Singular Value Decomposition (SVD)

# Step 1: Initialize missing values with mean
data_svd_init = data.copy()
data_svd_init[numeric_cols] = data_svd_init[numeric_cols].fillna(
    data_svd_init[numeric_cols].mean()
)

# Step 2: Apply SVD with 2 components
svd = TruncatedSVD(n_components=2)
X_reduced = svd.fit_transform(data_svd_init[numeric_cols])

# Step 3: Reconstruct the matrix
X_reconstructed = svd.inverse_transform(X_reduced)

# Step 4: Replace missing values with reconstructed values ONLY at NaN positions
data_svd_imputed = data.copy()

for j, col in enumerate(numeric_cols):
    # Find missing values
    missing_mask = data_svd_imputed[col].isna()

    # Replace only missing positions with reconstructed values
    data_svd_imputed.loc[missing_mask, col] = X_reconstructed[missing_mask, j]

print("SVD Imputation (Fixed):")
display(data_svd_imputed)


SVD Imputation (Fixed):


,feature_1,feature_2,feature_3,group
0,1.000000,2.500000,10.000000,A
1,2.000000,3.964261,11.000000,A
2,2.782491,3.500000,12.000000,B
3,4.000000,4.500000,12.312214,B
4,5.000000,5.500000,14.000000,B
5,3.261897,6.500000,15.000000,A


8. EM Algorithm-Based Imputation (Iterative Imputer)

The EM algorithm iteratively estimates missing values by:

E-step (Expectation):
Estimate missing values using current model parameters.

M-step (Maximization):
Update model parameters using the newly filled dataset.

This repeats until convergence.

In Python, the IterativeImputer from scikit-learn performs EM-style imputation using a regression model (Bayesian Ridge) to predict missing values.

In [8]:
# Method 6: EM Algorithm-Based Imputation (IterativeImputer)

# Create a copy of the data
data_em = data.copy()

# Initialize EM-style imputer (Bayesian Ridge as estimator)
em_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=10,
    imputation_order='ascending',
    random_state=0
)

# Only apply to numeric columns
X_em = em_imputer.fit_transform(data_em[numeric_cols])

# Replace numeric columns with imputed output
data_em[numeric_cols] = X_em

print("EM Algorithm-Based Imputation:")
display(data_em)


EM Algorithm-Based Imputation:


,feature_1,feature_2,feature_3,group
0,1.000000,2.500000,10.000000,A
1,2.000000,3.050006,11.000000,A
2,3.011264,3.500000,12.000000,B
3,4.000000,4.500000,12.990861,B
4,5.000000,5.500000,14.000000,B
5,5.996394,6.500000,15.000000,A


9. Comparison of All Imputation Methods

In this section, we compare the results of all six imputation techniques side-by-side:

Single Value (Mean)

Single Value (Median)

Group Mean

k-NN

Partial Mean

SVD

EM Algorithm

We evaluate each method using Reconstruction Error, which measures how close imputed values are compared to the initial mean-filled reference dataset.

This provides a quantitative comparison across methods.

In [9]:
# Prepare all imputed versions in a dictionary for comparison

datasets = {
    "Mean Imputation": data_mean[numeric_cols],
    "Median Imputation": data_median[numeric_cols],
    "Group Mean": data_group[numeric_cols],
    "kNN Imputation": data_knn[numeric_cols],
    "Partial Mean": data_partial_imputed[numeric_cols],
    "SVD Imputation": data_svd_imputed[numeric_cols],
    "EM Imputation": data_em[numeric_cols]
}

print("Datasets collected for comparison.")


Datasets collected for comparison.


In [10]:
# We need a reference dataset without NaN to compute real error.
# Since we don't have ground truth, we use mean-imputed data as baseline.

reference = data_mean[numeric_cols].values

errors = {}

for name, df in datasets.items():
    diff = reference - df.values
    mse = np.mean(diff ** 2)
    errors[name] = mse

errors


{'Mean Imputation': np.float64(0.0),
 'Median Imputation': np.float64(0.008888888888888904),
 'Group Mean': np.float64(0.26999999999999996),
 'kNN Imputation': np.float64(0.10518518518518516),
 'Partial Mean': np.float64(0.022777777777777793),
 'SVD Imputation': np.float64(0.02281237650924999),
 'EM Imputation': np.float64(0.635005697433194)}

In [11]:
# Display reconstruction errors in a sorted table

comparison_df = pd.DataFrame.from_dict(errors, orient="index", columns=["Reconstruction Error"])
comparison_df = comparison_df.sort_values(by="Reconstruction Error")

print("Comparison of Imputation Methods:")
comparison_df


Comparison of Imputation Methods:


,Reconstruction Error
Mean Imputation,0.000000
Median Imputation,0.008889
Partial Mean,0.022778
SVD Imputation,0.022812
kNN Imputation,0.105185
Group Mean,0.270000
EM Imputation,0.635006


10. Conclusion

In this notebook, we implemented and compared six different methods for handling missing data:

Single Value Imputation (Mean/Median)

Group-Based Imputation

k-Nearest Neighbors (k-NN)

Partial Mean Imputation

Singular Value Decomposition (SVD)

EM Algorithm-Based Imputation

Summary of Findings

Simple methods (mean/median) are fast but ignore feature relationships.

Group-based imputation captures structure within categories.

k-NN leverages similarity between samples and improves accuracy.

Partial mean avoids outlier distortion.

SVD reconstructs missing values using latent structure.

EM Algorithm produces the most statistically consistent imputations.

Final Notes

This notebook serves as the complete programming portion of the CS 509 final project.
All methods have been implemented, demonstrated on an example dataset, and compared quantitatively.

For detailed theory, mathematical models, and diagrams, refer to the PowerPoint presentation.